# Whisper IPO Training

This notebook fine-tunes a Whisper checkpoint with Identity Preference Optimization (IPO). It reads `dataset-ipo_20261023.json`, where each row supplies an audio path and preferred/rejected transcripts. Empty or identical transcript pairs are skipped; checkpoint selection uses validation WER, followed by a held-out test evaluation.

Training is deliberately in its own cell. Run setup and inspect the configuration first; the training cell loads two copies of the model and can require substantial GPU memory.

In [ ]:
# Training configuration
# Edit these values before training. The defaults match the local dataset and checkpoint used by the trainer script.
# `MAX_SAMPLES` can limit data for a small smoke run, but must be at least three.
  
from pathlib import Path

BASE_MODEL_PATH     = "/home/luca/Development/chiara-speech2text_models/chiara_whisper_20261026"

DATASET_PATH        = Path("/home/luca/Development/chiara-speech2text_data/dataset-ipo_20261023.json")
MODELS_OUTPUT_DIR   = Path("/home/luca/Development/chiara-speech2text_models/temp")

OUTPUT_DIR          = None  # None creates a dated chiara_whisper_ipo folder under MODELS_OUTPUT_DIR
AUDIO_ROOT          = None  # Set this only when audio_data contains relative paths

LANGUAGE                    = "it"
TASK                        = "transcribe"
N_MEL_BINS                  = 128

EPOCHS                      = 2
TRAIN_BATCH_SIZE            = 6
EVAL_BATCH_SIZE             = 4
WARMUP_STEPS                = 10
EVAL_STEPS                  = 20
MAX_GENERATION_LENGTH       = 48
EARLY_STOPPING_PATIENCE     = 30
EARLY_STOPPING_THRESHOLD    = 0.025

VALIDATION_RATIO            = 0.1
TEST_RATIO                  = 0.1

SEED                        = 42
MAX_SAMPLES                 = None

LEARNING_RATE               = 5e-7 #1e-6 
IPO_BETA                    = 0.2
#IPO_LOSS_WEIGHT             = 0.1
IPO_LOSS_WEIGHT             = 0
CHOSEN_CE_WEIGHT            = 10.0

print(f"Dataset:    {DATASET_PATH}")
print(f"Base model: {BASE_MODEL_PATH}")
print(f"IPO beta: {IPO_BETA}; IPO loss weight: {IPO_LOSS_WEIGHT}; chosen CE weight: {CHOSEN_CE_WEIGHT}; epochs: {EPOCHS}; batch size: {TRAIN_BATCH_SIZE}")

## 1. Set Notebook Kernel to Python

Use the project's Python environment (`.stt`) as the notebook kernel. The notebook metadata is set to the Python 3 kernel; confirm the selected kernel in VS Code before running code cells.

## 2. Import Python Libraries

These are the training and data-processing dependencies used by the script version. Install any missing packages in the selected `.stt` environment before continuing.

In [ ]:
import copy
import json
import os
import random
import time
from datetime import datetime
from typing import Any

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import numpy as np
import soundfile as sf
import torch
from jiwer import wer
from torch.utils.data import Dataset
from transformers import (
    EarlyStoppingCallback,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    WhisperForConditionalGeneration,
    WhisperProcessor,
    set_seed,
)

print(f"Python: {os.sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}; CUDA available: {torch.cuda.is_available()}")

In [ ]:
# disable tokenizer parallelism to avoid deadlocks on linux

os.environ["TOKENIZERS_PARALLELISM"] = "false"

## 3. Write and Run Python Code Cells

The next cells define the data-loading and splitting functions, then run them on the IPO JSON. Each usable record must have a readable audio file plus different non-empty winner and loser transcripts.

In [ ]:
def resolve_audio_path(value: str, dataset_path: Path, audio_root: Path | None) -> Path:
    audio_path = Path(value).expanduser()
    if audio_path.is_absolute():
        return audio_path
    if audio_root is not None:
        rooted_path = audio_root.expanduser() / audio_path
        if rooted_path.is_file():
            return rooted_path
    return dataset_path.parent / audio_path


def load_preference_rows(
    dataset_path: Path,
    audio_root: Path | None = None,
    max_samples: int | None = None,
) -> tuple[list[dict[str, Any]], int]:
    with dataset_path.open("r", encoding="utf-8") as handle:
        rows = json.load(handle)
    if not isinstance(rows, list):
        raise ValueError("IPO dataset must be a JSON array of records.")

    usable_rows = []
    skipped_pairs = 0
    for row_number, row in enumerate(rows, start=1):
        chosen = str(row.get("ipo_transcript_winner") or "").strip()
        rejected = str(row.get("ipo_transcript_loser") or "").strip()
        if not chosen or not rejected or chosen.casefold() == rejected.casefold():
            skipped_pairs += 1
            continue

        audio_value = str(row.get("audio_data") or "").strip()
        if not audio_value:
            raise ValueError(f"Record {row_number} has no audio_data path.")
        audio_path = resolve_audio_path(audio_value, dataset_path, audio_root)
        if not audio_path.is_file():
            raise FileNotFoundError(f"Audio file for record {row_number} was not found: {audio_path}")
        usable_rows.append({"audio_path": audio_path, "chosen": chosen, "rejected": rejected})

    if max_samples is not None:
        if max_samples < 1:
            raise ValueError("MAX_SAMPLES must be greater than zero.")
        usable_rows = usable_rows[:max_samples]
    if len(usable_rows) < 3:
        raise ValueError("At least three usable preference pairs are required.")
    return usable_rows, skipped_pairs


def split_rows(rows, validation_ratio: float, test_ratio: float, seed: int):
    if validation_ratio <= 0 or test_ratio <= 0 or validation_ratio + test_ratio >= 1:
        raise ValueError("Validation and test ratios must be positive and sum to less than 1.")

    shuffled_rows = list(rows)
    random.Random(seed).shuffle(shuffled_rows)
    validation_count = max(1, round(len(rows) * validation_ratio))
    test_count = max(1, round(len(rows) * test_ratio))
    train_count = len(rows) - validation_count - test_count
    if train_count < 1:
        raise ValueError("Dataset is too small for the requested split ratios.")

    train_end = train_count
    validation_end = train_end + validation_count
    return (
        shuffled_rows[:train_end],
        shuffled_rows[train_end:validation_end],
        shuffled_rows[validation_end:],
    )

In [ ]:
preference_rows, skipped_pair_count = load_preference_rows(
    DATASET_PATH,
    audio_root=AUDIO_ROOT,
    max_samples=MAX_SAMPLES,
)
train_rows, validation_rows, test_rows = split_rows(
    preference_rows,
    validation_ratio    = VALIDATION_RATIO,
    test_ratio          = TEST_RATIO,
    seed                = SEED,
)
set_seed(SEED)

print(f"Usable pairs: {len(preference_rows)}; skipped empty/equal pairs: {skipped_pair_count}")
print(f"Splits: train={len(train_rows)}, validation={len(validation_rows)}, test={len(test_rows)}")

### Audio features, target tokens, and padding

Audio features are computed lazily as batches are requested. Winner and loser token sequences are padded to the same batch-wide length so the trainer can score both sides together.

In [ ]:
class WhisperIPODataset(Dataset):
    def __init__(self, rows, processor: WhisperProcessor):
        self.rows = rows
        self.processor = processor
        self.tokenized_rows = [
            (
                processor.tokenizer(row["chosen"]).input_ids,
                processor.tokenizer(row["rejected"]).input_ids,
            )
            for row in rows
        ]

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index: int) -> dict[str, Any]:
        row = self.rows[index]
        audio, sample_rate = sf.read(row["audio_path"], dtype="float32")
        if audio.ndim > 1:
            audio = audio.mean(axis=1)

        target_sample_rate = self.processor.feature_extractor.sampling_rate
        if sample_rate != target_sample_rate:
            import librosa

            audio = librosa.resample(
                audio,
                orig_sr=sample_rate,
                target_sr=target_sample_rate,
            )

        features = self.processor.feature_extractor(
            audio,
            sampling_rate=target_sample_rate,
        ).input_features[0]
        chosen_ids, rejected_ids = self.tokenized_rows[index]
        return {
            "input_features": features,
            "chosen_labels": chosen_ids,
            "rejected_labels": rejected_ids,
        }


class DataCollatorSpeechPreference:
    def __init__(self, processor: WhisperProcessor):
        self.processor = processor
        self.decoder_start_token_id = processor.tokenizer.convert_tokens_to_ids(
            "<|startoftranscript|>"
        )

    def __call__(self, features: list[dict[str, Any]]) -> dict[str, torch.Tensor]:
        input_features = [
            {"input_features": feature["input_features"]} for feature in features
        ]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        max_label_length = max(
            max(len(feature["chosen_labels"]) for feature in features),
            max(len(feature["rejected_labels"]) for feature in features),
        )
        chosen_batch = self.processor.tokenizer.pad(
            [{"input_ids": feature["chosen_labels"]} for feature in features],
            padding="max_length",
            max_length=max_label_length,
            return_tensors="pt",
        )
        rejected_batch = self.processor.tokenizer.pad(
            [{"input_ids": feature["rejected_labels"]} for feature in features],
            padding="max_length",
            max_length=max_label_length,
            return_tensors="pt",
        )
        chosen_labels = chosen_batch["input_ids"].masked_fill(
            chosen_batch["attention_mask"].ne(1), -100
        )
        rejected_labels = rejected_batch["input_ids"].masked_fill(
            rejected_batch["attention_mask"].ne(1), -100
        )
        if (chosen_labels[:, 0] == self.decoder_start_token_id).all().item():
            chosen_labels = chosen_labels[:, 1:]
        if (rejected_labels[:, 0] == self.decoder_start_token_id).all().item():
            rejected_labels = rejected_labels[:, 1:]

        batch["chosen_labels"] = chosen_labels
        batch["rejected_labels"] = rejected_labels
        batch["labels"] = chosen_labels
        return batch

In [ ]:
processor = WhisperProcessor.from_pretrained(
    BASE_MODEL_PATH,
    language=LANGUAGE,
    task=TASK,
    num_mel_bins=N_MEL_BINS,
)

sample_dataset = WhisperIPODataset(preference_rows[:1], processor)
sample_batch = DataCollatorSpeechPreference(processor)([sample_dataset[0]])
print("One audio feature batch:", tuple(sample_batch["input_features"].shape))
print("Chosen/rejected label shapes:", tuple(sample_batch["chosen_labels"].shape), tuple(sample_batch["rejected_labels"].shape))

## IPO objective and trainer

For each audio sample, the policy and frozen reference score both transcripts. The IPO loss regresses the chosen-minus-rejected log-ratio toward $1/(2\beta)$, while a weighted cross-entropy term directly trains the policy on the chosen transcript. Validation WER is computed against that chosen transcript.

In [ ]:
def sequence_log_probabilities(logits: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
    mask = labels.ne(-100)
    safe_labels = labels.masked_fill(~mask, 0)
    token_log_probabilities = torch.log_softmax(logits.float(), dim=-1)
    selected_log_probabilities = token_log_probabilities.gather(
        2,
        safe_labels.unsqueeze(-1),
    ).squeeze(-1)
    return (selected_log_probabilities * mask).sum(dim=1)


class WhisperIPOTrainer(Seq2SeqTrainer):
    def __init__(
        self,
        *args: Any,
        reference_model: WhisperForConditionalGeneration,
        beta: float,
        ipo_loss_weight: float = 1.0,
        chosen_ce_weight: float = 1.0,
        **kwargs: Any,
    ):
        super().__init__(*args, **kwargs)
        self.reference_model = reference_model
        self.beta = beta
        self.ipo_loss_weight = ipo_loss_weight
        self.chosen_ce_weight = chosen_ce_weight

    def compute_loss(
        self,
        model: WhisperForConditionalGeneration,
        inputs: dict[str, torch.Tensor],
        return_outputs: bool = False,
        num_items_in_batch: int | None = None,
    ) -> torch.Tensor | tuple[torch.Tensor, dict[str, torch.Tensor]]:
        del num_items_in_batch
        chosen_labels = inputs["chosen_labels"]
        rejected_labels = inputs["rejected_labels"]
        labels = torch.cat((chosen_labels, rejected_labels), dim=0)
        input_features = torch.cat(
            (inputs["input_features"], inputs["input_features"]),
            dim=0,
        )

        policy_logits = model(input_features=input_features, labels=labels).logits
        with torch.no_grad():
            reference_logits = self.reference_model(
                input_features=input_features,
                labels=labels,
            ).logits

        policy_logps = sequence_log_probabilities(policy_logits, labels)
        reference_logps = sequence_log_probabilities(reference_logits, labels)
        log_ratios = policy_logps - reference_logps
        chosen_log_ratio, rejected_log_ratio = log_ratios.chunk(2)
        preference_logits = chosen_log_ratio - rejected_log_ratio
        ipo_loss = (preference_logits - 1.0 / (2.0 * self.beta)).square().mean()

        chosen_batch_size = chosen_labels.shape[0]
        chosen_ce_loss = torch.nn.functional.cross_entropy(
            policy_logits[:chosen_batch_size].reshape(-1, policy_logits.shape[-1]),
            chosen_labels.reshape(-1),
            ignore_index=-100,
        )
        loss = self.ipo_loss_weight * ipo_loss + self.chosen_ce_weight * chosen_ce_loss
        if return_outputs:
            return loss, {"logits": policy_logits}
        return loss

    def prediction_step(
        self,
        model: WhisperForConditionalGeneration,
        inputs: dict[str, torch.Tensor],
        prediction_loss_only: bool,
        ignore_keys: list[str] | None = None,
        **gen_kwargs: Any,
    ):
        del ignore_keys
        inputs = self._prepare_inputs(inputs)
        with torch.no_grad():
            loss = self.compute_loss(model, inputs)
        if prediction_loss_only:
            return loss.detach(), None, None

        generation_length = gen_kwargs.get("max_length", self.args.generation_max_length)
        generated_tokens = model.generate(
            input_features=inputs["input_features"],
            max_length=generation_length,
        )
        labels = inputs["chosen_labels"]
        if generated_tokens.shape[-1] < generation_length:
            generated_tokens = self._pad_tensors_to_max_len(generated_tokens, generation_length)
        if labels.shape[-1] < generation_length:
            labels = self._pad_tensors_to_max_len(labels, generation_length)
        return loss.detach(), generated_tokens, labels


def build_compute_metrics(processor: WhisperProcessor):
    def compute_metrics(prediction: Any) -> dict[str, float]:
        predictions = prediction.predictions
        if isinstance(predictions, tuple):
            predictions = predictions[0]
        label_ids = np.where(
            prediction.label_ids == -100,
            processor.tokenizer.pad_token_id,
            prediction.label_ids,
        )
        predicted_text = processor.tokenizer.batch_decode(predictions, skip_special_tokens=True)
        reference_text = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)
        return {"wer": 100.0 * wer(reference_text, predicted_text)}

    return compute_metrics

## Load policy and reference models

The reference is a frozen copy of the starting checkpoint. This doubles model memory use; reduce batch size if the GPU runs out of memory.

In [ ]:
if IPO_BETA <= 0:
    raise ValueError("IPO_BETA must be greater than zero.")

processor = WhisperProcessor.from_pretrained(
    BASE_MODEL_PATH,
    language     = LANGUAGE,
    task         = TASK,
    num_mel_bins = N_MEL_BINS,
)
policy_model = WhisperForConditionalGeneration.from_pretrained(BASE_MODEL_PATH).to(torch.float32)
policy_model.generation_config.language             = LANGUAGE
policy_model.generation_config.task                 = TASK
policy_model.generation_config.forced_decoder_ids   = None
policy_model.config.forced_decoder_ids              = None

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
policy_model = policy_model.to(device)
reference_model = copy.deepcopy(policy_model)
reference_model.requires_grad_(False)
reference_model.eval()

print("Policy device:   ", next(policy_model.parameters()).device)
print("Reference device:", next(reference_model.parameters()).device)

### Trainer configuration

Checkpoint saves are aligned with validation evaluations, and the best checkpoint is selected by validation WER.

In [ ]:
if OUTPUT_DIR is None:
    output_dir = MODELS_OUTPUT_DIR / f"chiara_whisper_ipo_{datetime.now():%Y%m%d}"
else:
    output_dir = Path(OUTPUT_DIR).expanduser()
if output_dir.exists():
    suffix = 1
    while output_dir.with_name(f"{output_dir.name}_{suffix}").exists():
        suffix += 1
    output_dir = output_dir.with_name(f"{output_dir.name}_{suffix}")
output_dir.mkdir(parents=True)
trainer_output_dir = output_dir / "trainer"

eval_steps = max(1, EVAL_STEPS)
training_args = Seq2SeqTrainingArguments(
    output_dir                  = str(trainer_output_dir),
    per_device_train_batch_size = TRAIN_BATCH_SIZE,
    per_device_eval_batch_size  = EVAL_BATCH_SIZE,
    gradient_accumulation_steps = 1,
    gradient_checkpointing      = False,
    fp16                        = False,
    num_train_epochs            = EPOCHS,
    learning_rate               = LEARNING_RATE,
    lr_scheduler_type           = "constant_with_warmup",
    warmup_steps                = WARMUP_STEPS,
    eval_strategy               = "steps",
    eval_steps                  = eval_steps,
    predict_with_generate       = True,
    generation_max_length       = MAX_GENERATION_LENGTH,
    metric_for_best_model       = "eval_wer",
    greater_is_better           = False,
    save_strategy               = "steps",
    save_steps                  = eval_steps,
    save_total_limit            = 2,
    load_best_model_at_end      = True,
    logging_steps               = 1,
    remove_unused_columns       = False,
    push_to_hub                 = False,
    dataloader_num_workers      = 0, # above 0, we cause a deadlock
    dataloader_pin_memory       = torch.cuda.is_available(),
    dataloader_persistent_workers= False,
    report_to                   = ["tensorboard"],
)

trainer = WhisperIPOTrainer(
    args             = training_args,
    model            = policy_model,
    reference_model  = reference_model,
    beta             = IPO_BETA,
    ipo_loss_weight  = IPO_LOSS_WEIGHT,
    chosen_ce_weight = CHOSEN_CE_WEIGHT,
    train_dataset    = WhisperIPODataset(train_rows, processor),
    eval_dataset     = WhisperIPODataset(validation_rows, processor),
    data_collator    = DataCollatorSpeechPreference(processor),
    processing_class = processor,
    compute_metrics  = build_compute_metrics(processor),
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=EARLY_STOPPING_PATIENCE,
            early_stopping_threshold=EARLY_STOPPING_THRESHOLD,
        )
    ],
)
print(f"Training output: {output_dir}")

## Train and evaluate

Run this cell to start training. The test split is evaluated after training and is not used for checkpoint selection.

In [ ]:
# Start tensorboard
# Tensorboard allows to show the results from the next cell. To use it:
# - Play this cell, it will show an empty output
# - Open http://localhost:6006 in your browser
# NOTE: if ssh'd, you have to ensure that port 6006 is forwarded (e.g. go to the "Ports" tab in Visual Code, and you have to manually forward port 6006)

%load_ext tensorboard
%reload_ext tensorboard
%tensorboard --logdir {trainer_output_dir}

In [ ]:
# Establish the starting validation WER before any optimizer updates.
baseline_metrics = trainer.evaluate(
    eval_dataset=WhisperIPODataset(validation_rows, processor),
    metric_key_prefix="eval",
)
baseline_wer = baseline_metrics["eval_wer"]
print("Baseline validation metrics:", baseline_metrics)

In [ ]:
# train
start_time              = time.perf_counter()
train_result            = trainer.train()
training_duration_hours = (time.perf_counter() - start_time) / 3600.0

In [ ]:
best_training_wer = trainer.state.best_metric
selected_model = "trained"
if best_training_wer is None or best_training_wer >= baseline_wer:
    print(
        f"No trained checkpoint beat baseline WER ({baseline_wer:.2f}); "
        "restoring the starting model."
    )
    trainer.model.load_state_dict(reference_model.state_dict())
    selected_model = "base"

In [ ]:
# evaluate
test_metrics = trainer.evaluate(
    WhisperIPODataset(test_rows, processor),
    metric_key_prefix="test",
)

# print results
print("Best trained validation WER:", best_training_wer)
print("Selected model:", selected_model)
print("Training result:", train_result.metrics)
print("Held-out test metrics:", test_metrics)

## Save model and metadata

This saves the selected best model, processor, trainer state, test metrics, and reproducibility metadata locally. It does not upload anything to the Hugging Face Hub.

In [ ]:
trainer.save_model(str(output_dir))
processor.save_pretrained(str(output_dir))
trainer.save_metrics("test", test_metrics)
trainer.save_state()

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "n/a"
metadata = {
    "base_model":               BASE_MODEL_PATH,
    "selected_model":           selected_model,
    "baseline_validation_wer":  baseline_wer,
    "best_training_validation_wer": best_training_wer,
    "training_technique":       "Identity Preference Optimization (IPO) with chosen-transcript CE",
    "training_date":            datetime.now().strftime("%Y%m%d"),
    "training_duration_hours":  round(training_duration_hours, 2),
    "dataset":                  str(DATASET_PATH),
    "dataset_pairs":            len(preference_rows),
    "dataset_pairs_skipped":    skipped_pair_count,
    "split_sizes": {
        "train":        len(train_rows),
        "validation":   len(validation_rows),
        "test":         len(test_rows),
    },
    "language":          LANGUAGE,
    "task":              TASK,
    "n_mel_bins":        N_MEL_BINS,
    "ipo_beta":          IPO_BETA,
    "ipo_loss_weight":   IPO_LOSS_WEIGHT,
    "chosen_ce_weight":  CHOSEN_CE_WEIGHT,
    "batch_size":        TRAIN_BATCH_SIZE,
    "learning_rate":     LEARNING_RATE,
    "epochs":            EPOCHS,
    "seed":              SEED,
    "gpu_name":          gpu_name,
    "test_metrics":      test_metrics,
}

metadata_dir = output_dir / "metadata"
metadata_dir.mkdir(exist_ok=True)
with (metadata_dir / "metadata_model.json").open("w", encoding="utf-8") as handle:
    json.dump(metadata, handle, indent=4, ensure_ascii=False)
with (metadata_dir / "trainer_parameters.json").open("w", encoding="utf-8") as handle:
    json.dump(training_args.to_dict(), handle, indent=4, default=str)

print("Test metrics:", test_metrics)
print("Model and metadata saved to:", output_dir)